In [43]:
!nvidia-smi

Sat Jul 26 02:19:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 553.24                 Driver Version: 553.24         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:02:00.0 Off |                  Off |
| 30%   56C    P8             18W /  300W |     972MiB /  49140MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Setup environment

In [37]:
# sync python module
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup config

In [71]:
import os

workspace_dir = '/nfs/Workspace/CardiacSegV2'
model_name = 'testnet' #unet3d attention_unet DynUNet cotr unetr swinunetr unetcnx_a1 unest testnet #baseline baseline_rescbam baseline_inceptionnext
data_name = 'chgh'
sub_data_dir_name = 'new80'
exp_name = 'exp_newchgh_9_3_3_fold1' 
data_dict_file_name = 'exp_newchgh_9_3_3_fold1.json' 


tune_mode = 'train'

# set exp dir
root_exp_dir = os.path.join(
    workspace_dir, 
    'exps',
    'exps',
    model_name,
    data_name,
    'tune_results'
)

# set data dir
root_data_dir = os.path.join(
    workspace_dir, 
    'dataset',
    data_name
)
data_dir = os.path.join(root_data_dir, sub_data_dir_name)

# data dict json path
data_dicts_json = os.path.join(workspace_dir, 'exps', 'data_dicts', data_name, data_dict_file_name)

# set model, log, eval dir
model_dir = os.path.join('./', 'models')
log_dir = os.path.join('./', 'logs')
eval_dir = os.path.join('./', 'evals')

# model path
best_checkpoint = os.path.join(model_dir, 'best_model.pth')
final_checkpoint = os.path.join(model_dir, 'final_model.pth')

# mkdir root exp dir
os.makedirs(root_exp_dir, exist_ok=True)

# for pretrain
pretrain_exp_name = 'exp_50'
pretrain_data_name = 'image_cas'
pretrain_model_dir = os.path.join(
    workspace_dir,
    'exps',
    'exps',
    model_name,
    pretrain_data_name,
    'pretrain',
    pretrain_exp_name,
    'models'
)
pretrain_checkpoint = os.path.join(pretrain_model_dir, 'model_bestValRMSE.pt')

%cd {root_exp_dir}/../

e:\nfs\Workspace\CardiacSegV2\exps\exps\testnet\chgh


## Train TestNet

In [ ]:
# training
!set PYTHONPATH={workspace_dir} && \
python {workspace_dir}/expers/tune.py \
--tune_mode={tune_mode} \
--exp_name={exp_name} \
--data_name={data_name} \
--data_dir={data_dir} \
--root_exp_dir={root_exp_dir} \
--model_name={model_name}\
--model_dir={model_dir} \
--log_dir={log_dir} \
--eval_dir={eval_dir} \
--start_epoch=0 \
--val_every=20 \
--max_early_stop_count=20 \
--max_epoch=8000  \
--data_dicts_json={data_dicts_json} \
--pin_memory \
--out_channels=2 \
--patch_size=2 \
--feature_size=48 \
--drop_rate=0.1 \
--depths 3 3 9 3 \
--kernel_size 7 \
--exp_rate 4 \
--norm_name='layer' \
--a_min=-42 \
--a_max=423 \
--space_x=0.7 \
--space_y=0.7 \
--space_z=1.0 \
--roi_x=128 \
--roi_y=128 \
--roi_z=128 \
--optim="AdamW" \
--lr=2e-3 \
--weight_decay=5e-4 \
--checkpoint={final_checkpoint} \
--use_init_weights \
--infer_post_process \
--deep_sup \
--save_eval_csv

== Status ==
Current time: 2025-08-04 23:37:12 (running for 00:00:00.18)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\testnet\chgh\tune_results\exp_newchgh_9_3_3_fold1
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+
| Trial name       | status   | loc             | exp                  |
|------------------+----------+-----------------+----------------------|
| main_e4241_00000 | RUNNING  | 127.0.0.1:10836 | {'exp': 'exp_ne_6f40 |
+------------------+----------+-----------------+----------------------+


(func pid=10836) a_max 423.0
(func pid=10836) a_min -42.0
(func pid=10836) space_x 0.7
(func pid=10836) roi_x 128
(func pid=10836) lr 0.002
(func pid=10836) weight_decay 0.0005
(func pid=10836) warmup_epochs 50
(func pid=10836) max_epochs 8000
(func pid=10836) cuda is available
(func pid=10836) model: testnet
(func pid=10836) patch size: 

2025-08-04 23:37:09,810	INFO worker.py:1625 -- Started a local Ray instance.
The `local_dir` argument of `Experiment is deprecated. Use `storage_path` or set the `TUNE_RESULT_DIR` environment variable instead.
2025-08-04 23:37:12,763	INFO tensorboardx.py:172 -- pip install "ray[tune]" to see TensorBoard files.
2025-08-04 23:37:12,763	WARNING callback.py:142 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`
2025-08-04 23:37:23,035	ERROR trial_runner.py:1450 -- Trial main_e4241_00000: Error happened when processing _ExecutorEventType.TRAINING_RESULT.
ray.exceptions.RayTaskError(ValueError): ray::ImplicitFunc.train() (pid=10836, ip=127.0.0.1, repr=func)
  File "python\ray\_raylet.pyx", line 877, in ray._raylet.execute_task
  File "python\ray\_raylet.pyx", line 881, in ray._raylet.execute_task
  File "python\ray\_ray

## Train UNETCNX

In [22]:
# training
#!PYTHONPATH={workspace_dir} /opt/conda/bin/python {workspace_dir}/expers/tune.py \
#!python {workspace_dir}/expers/tune.py \
!set PYTHONPATH={workspace_dir} && \
python {workspace_dir}/expers/tune.py \
--tune_mode={tune_mode} \
--exp_name={exp_name} \
--data_name={data_name} \
--data_dir={data_dir} \
--root_exp_dir={root_exp_dir} \
--model_name={model_name}\
--model_dir={model_dir} \
--log_dir={log_dir} \
--eval_dir={eval_dir} \
--start_epoch=0 \
--val_every=20 \
--max_early_stop_count=20 \
--max_epoch=8000  \
--data_dicts_json={data_dicts_json} \
--pin_memory \
--out_channels=2 \
--patch_size=4 \
--feature_size=48 \
--drop_rate=0.1 \
--depths 3 3 9 3 \
--kernel_size 7 \
--exp_rate 4 \
--norm_name='layer' \
--a_min=-42 \
--a_max=423 \
--space_x=0.7 \
--space_y=0.7 \
--space_z=1.0 \
--roi_x=128 \
--roi_y=128 \
--roi_z=128 \
--optim="AdamW" \
--lr=7e-4 \
--weight_decay=5e-4 \
--checkpoint={final_checkpoint} \
--use_init_weights \
--infer_post_process \
--deep_sup \
--save_eval_csv
#--test_mode \
#--batch_size=4 \
#--resume_tuner \


== Status ==
Current time: 2025-07-23 19:59:39 (running for 00:00:00.17)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\unet3d\chgh\tune_results\exp_newchgh_9_3_3_fold1
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+
| Trial name       | status   | loc             | exp                  |
|------------------+----------+-----------------+----------------------|
| main_83039_00000 | RUNNING  | 127.0.0.1:10572 | {'exp': 'exp_ne_9a80 |
+------------------+----------+-----------------+----------------------+


(func pid=10572) a_max 423.0
(func pid=10572) a_min -42.0
(func pid=10572) space_x 0.7
(func pid=10572) roi_x 128
(func pid=10572) lr 0.0007
(func pid=10572) weight_decay 0.0005
(func pid=10572) warmup_epochs 50
(func pid=10572) max_epochs 8000
(func pid=10572) cuda is available
(func pid=10572) model: unet3d
(func pid=10572) loss: dice ce

2025-07-23 19:59:36,837	INFO worker.py:1625 -- Started a local Ray instance.
The `local_dir` argument of `Experiment is deprecated. Use `storage_path` or set the `TUNE_RESULT_DIR` environment variable instead.
2025-07-23 19:59:39,804	INFO tensorboardx.py:172 -- pip install "ray[tune]" to see TensorBoard files.
2025-07-23 19:59:39,804	WARNING callback.py:142 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`
(func pid=10572) monai.transforms.io.dictionary LoadImaged.__init__:image_only: Current default value of argument `image_only=False` has been deprecated since version 1.1. It will be changed to `image_only=True` in version 1.3.
(func pid=10572) <class 'monai.transforms.utility.dictionary.AddChanneld'>: Class `AddChanneld` has been deprecated since version 0.8. It will be removed in version 1.3. please use MetaT

## Train other models

In [ ]:
# training
#!PYTHONPATH={workspace_dir} /opt/conda/bin/python {workspace_dir}/expers/tune.py \
#!python {workspace_dir}/expers/tune.py \
!set PYTHONPATH={workspace_dir} && \
python {workspace_dir}/expers/tune.py \
--tune_mode={tune_mode} \
--exp_name={exp_name} \
--data_name={data_name} \
--data_dir={data_dir} \
--root_exp_dir={root_exp_dir} \
--model_name={model_name}\
--model_dir={model_dir} \
--log_dir={log_dir} \
--eval_dir={eval_dir} \
--start_epoch=0 \
--val_every=20 \
--max_early_stop_count=20 \
--max_epoch=8000  \
--data_dicts_json={data_dicts_json} \
--pin_memory \
--out_channels=2 \
--patch_size=4 \
--feature_size=48 \
--drop_rate=0.1 \
--depths 3 3 9 3 \
--kernel_size 7 \
--exp_rate 4 \
--norm_name='layer' \
--a_min=-42 \
--a_max=423 \
--space_x=0.7 \
--space_y=0.7 \
--space_z=1.0 \
--roi_x=128 \
--roi_y=128 \
--roi_z=128 \
--optim="AdamW" \
--batch_size=1 \
--lr=1e-4 \
--weight_decay=5e-4 \
--checkpoint={final_checkpoint} \
--use_init_weights \
--infer_post_process \
--save_eval_csv
# --resume_tuner
# --test_mode

== Status ==
Current time: 2025-08-02 15:07:16 (running for 00:00:00.19)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\swinunetr\chgh\tune_results\exp_newchgh_9_3_3_fold4
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+
| Trial name       | status   | loc             | exp                  |
|------------------+----------+-----------------+----------------------|
| main_52633_00000 | RUNNING  | 127.0.0.1:11212 | {'exp': 'exp_ne_9040 |
+------------------+----------+-----------------+----------------------+


(func pid=11212) a_max 423.0
(func pid=11212) a_min -42.0
(func pid=11212) space_x 0.7
(func pid=11212) roi_x 128
(func pid=11212) lr 0.0001
(func pid=11212) weight_decay 0.0005
(func pid=11212) warmup_epochs 50
(func pid=11212) max_epochs 8000
(func pid=11212) cuda is available
(func pid=11212) model: swinunetr
(func pid=11212) loss: d

2025-08-02 15:07:13,355	INFO worker.py:1625 -- Started a local Ray instance.
The `local_dir` argument of `Experiment is deprecated. Use `storage_path` or set the `TUNE_RESULT_DIR` environment variable instead.
2025-08-02 15:07:16,277	INFO tensorboardx.py:172 -- pip install "ray[tune]" to see TensorBoard files.
2025-08-02 15:07:16,277	WARNING callback.py:142 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`
(func pid=11212) monai.transforms.io.dictionary LoadImaged.__init__:image_only: Current default value of argument `image_only=False` has been deprecated since version 1.1. It will be changed to `image_only=True` in version 1.3.
(func pid=11212) <class 'monai.transforms.utility.dictionary.AddChanneld'>: Class `AddChanneld` has been deprecated since version 0.8. It will be removed in version 1.3. please use MetaT

|------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------|
| main_52633_00000 | RUNNING  | 127.0.0.1:11212 | {'exp': 'exp_ne_9040 |         0 |         0 |      0.915327 |     0 |
+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+


== Status ==
Current time: 2025-08-02 15:22:49 (running for 00:15:32.74)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\swinunetr\chgh\tune_results\exp_newchgh_9_3_3_fold4
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+
| Trial name       | status   | loc             | exp                  |   tt_dice |   tt_hd95 |   val_bst_acc |   esc |
|------------------+----------+-----------------+----------------------+-----------+-----------+--

[Epoch 401] Training (3609 Steps) (loss=0.06497):  11%|█         | 1/9 [00:18<02:24, 18.08s/it]
(func pid=11212) 
[Epoch 401] Training (3610 Steps) (loss=0.09943):  11%|█         | 1/9 [00:19<02:24, 18.08s/it]
(func pid=11212) 
[Epoch 401] Training (3610 Steps) (loss=0.09943):  22%|██▏       | 2/9 [00:19<00:57,  8.27s/it]
(func pid=11212) 
[Epoch 401] Training (3611 Steps) (loss=0.13052):  22%|██▏       | 2/9 [00:20<00:57,  8.27s/it]
[Epoch 401] Training (3611 Steps) (loss=0.13052):  33%|███▎      | 3/9 [00:20<00:30,  5.08s/it]
(func pid=11212) 
[Epoch 401] Training (3612 Steps) (loss=0.07138):  33%|███▎      | 3/9 [00:22<00:30,  5.08s/it]
[Epoch 401] Training (3612 Steps) (loss=0.07138):  44%|████▍     | 4/9 [00:22<00:17,  3.59s/it]
(func pid=11212) 
[Epoch 401] Training (3613 Steps) (loss=0.06729):  44%|████▍     | 4/9 [00:23<00:17,  3.59s/it]
[Epoch 401] Training (3613 Steps) (loss=0.06729):  56%|█████▌    | 5/9 [00:23<00:11,  2.76s/it]
(func pid=11212) 
[Epoch 401] Training (3614 S



== Status ==
Current time: 2025-08-02 17:55:45 (running for 02:48:29.04)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\swinunetr\chgh\tune_results\exp_newchgh_9_3_3_fold4
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+
| Trial name       | status   | loc             | exp                  |   tt_dice |   tt_hd95 |   val_bst_acc |   esc |
|------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------|
| main_52633_00000 | RUNNING  | 127.0.0.1:11212 | {'exp': 'exp_ne_9040 |         0 |         0 |      0.955367 |     5 |
+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+


== Status ==
Current time: 2025-08-02 17:55:50 (running for 02:48:34.07)
Using FIFO scheduling a

+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+
| Trial name       | status   | loc             | exp                  |   tt_dice |   tt_hd95 |   val_bst_acc |   esc |
|------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------|
| main_52633_00000 | RUNNING  | 127.0.0.1:11212 | {'exp': 'exp_ne_9040 |         0 |         0 |      0.957871 |     1 |
+------------------+----------+-----------------+----------------------+-----------+-----------+---------------+-------+


== Status ==
Current time: 2025-08-02 19:35:16 (running for 04:28:00.68)
Using FIFO scheduling algorithm.
Logical resource usage: 1.0/12 CPUs, 1.0/1 GPUs
Result logdir: e:\nfs\Workspace\CardiacSegV2\exps\exps\swinunetr\chgh\tune_results\exp_newchgh_9_3_3_fold4
Number of trials: 1/1 (1 RUNNING)
+------------------+----------+-----------------+----------------------+-----------+-----------+--

## Analysis

In [ ]:
#!PYTHONPATH=/nfs/Workspace/CardiacSegV2 /opt/conda/bin/python /nfs/Workspace/CardiacSeg/expers/tune_anal.py \
#!python {workspace_dir}/expers/tune_anal.py \
--exp_name={exp_name} \
--local_dir={root_exp_dir}

'PYTHONPATH' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC
